In [1]:
"""
mo_pycistopic_impute_accessibility.ipynb

This script is used to impute the accessibility

authors: Roy Oelen, Martijn van der Werf

"""

'\nmo_pycistopic_impute_accessibility.ipynb\n\nThis script is used to impute the accessibility\n\nauthors: Roy Oelen, Martijn van der Werf\n\n'

In [2]:
###########
# imports #
###########

# object
from pycisTopic.cistopic_class import CistopicObject
import pickle
# imputation
from pycisTopic.diff_features import (
    impute_accessibility,
    normalize_scores,
    find_highly_variable_features,
    find_diff_features
)

/home/umcg-roelen/miniconda3/envs/pycistopic_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-02-28 12:35:31,960	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [3]:
#############
# functions #
#############

# method to create md5
def create_md5_file(input_file):
    """
    Creates an MD5 hash of the specified file and writes it to a new file with the same name but .md5 added to the extension.

    Args:
        input_file (str): The path to the input file for which the MD5 hash should be created.

    Returns:
        int: Returns 0 on success, 1 on failure.

    Raises:
        FileNotFoundError: Thrown if the input file does not exist.
        IOError: Thrown if there is an error reading the input file (like permission denied) or writing the output md5 file.
    """
    try:
        # get an md5 of the file
        digest = None
        with open(input_file, "rb") as f:
            # try Python 3.11+ method if it is available
            if callable(getattr(hashlib, 'file_digest', None)):
                # digest with one command
                digest = hashlib.file_digest(f, 'md5')
            # or the older 3.8+ method if we don't have the newer method
            else:
                # initialize digest
                digest = hashlib.md5()
                # read file in chunks
                while chunk := f.read(8192):
                    # update digestion
                    digest.update(chunk)       
        # get the output path of the md5
        output_md5_loc = ''.join([input_file, '.md5'])
        # and write that
        with open(output_md5_loc, "w") as m:
            m.write(digest.hexdigest())
        # upon success, return 0
        return 0
    except Exception as e:
        print(f"Exception occured upon md5 file creation: {e}")
        return 1

In [4]:
##############################
# read the pycistopic object #
##############################

# location to store the object
pycistopic_object_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/objects/merged_major_and_minor_celltypes_120topics.pkl'
# use symlinks due to path size limitations
pycistopic_object_loc = './120'

# save the object
with open(pycistopic_object_loc, 'rb') as f:
   cistopic_obj = pickle.load(f)


In [5]:
######################
# perform imputation #
######################

# impute regions
imputed_acc_obj = impute_accessibility(
    cistopic_obj,
    selected_cells=None,
    selected_regions=None,
    scale_factor=10**6
)

2025-02-28 12:37:03,005 cisTopic     INFO     Imputing region accessibility
2025-02-28 12:37:03,007 cisTopic     INFO     Impute region accessibility for regions 0-20000
2025-02-28 12:37:50,841 cisTopic     INFO     Impute region accessibility for regions 20000-40000
2025-02-28 12:38:38,269 cisTopic     INFO     Impute region accessibility for regions 40000-60000
2025-02-28 12:39:25,621 cisTopic     INFO     Impute region accessibility for regions 60000-80000
2025-02-28 12:40:12,329 cisTopic     INFO     Impute region accessibility for regions 80000-100000
2025-02-28 12:40:59,366 cisTopic     INFO     Impute region accessibility for regions 100000-120000
2025-02-28 12:41:46,304 cisTopic     INFO     Impute region accessibility for regions 120000-140000
2025-02-28 12:42:34,323 cisTopic     INFO     Impute region accessibility for regions 140000-160000
2025-02-28 12:43:24,604 cisTopic     INFO     Impute region accessibility for regions 160000-180000
2025-02-28 12:44:11,993 cisTopic     

In [ ]:
###########################
# save imputation results #
###########################

# location to store the object
pycistopic_object_wimputations_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/objects/merged_major_and_minor_celltypes_120topics_imputed.pkl'

# save the object
with open(pycistopic_object_wimputations_loc, 'wb') as f:
   pickle.dump(imputed_acc_obj, f)

# make a checksum
create_md5_file(pycistopic_object_wimputations_loc)

In [ ]:
###########################
# normalize accessibility #
###########################

# normalize object
normalized_imputed_acc_obj = normalize_scores(imputed_acc_obj, scale_factor=10**4)

In [ ]:
#########################################
# get differentially accessible regions #
#########################################

# calculate variable regions
variable_regions = find_highly_variable_features(
    normalized_imputed_acc_obj,
    min_disp = 0.05,
    min_mean = 0.0125,
    max_mean = 3,
    max_disp = np.inf,
    n_bins=20,
    n_top_features=None,
    plot=True
)